# D.R.O.N.A. — 06 · Gesture Policies with LeRobot (ACT · Diffusion · SmolVLA)

Follows the **current official LeRobot docs (v0.6.x)**. Separate from the LLM
notebook (04) so their dependencies never clash.

Two paths:

* **Part A — state-only** (default, correct for pure gestures). A greet/nod/point
  motion doesn't depend on the visual scene, so the policy maps *joint state -> joint
  action*. No camera => no TorchCodec/PyAV/ffmpeg (the stack that broke earlier runs).
* **Part B — visuomotor (camera)**. Adds a camera observation so you can train
  **vision-conditioned ACT** and **fine-tune SmolVLA with LoRA/PEFT**. Camera frames
  are stored as `dtype="image"` (PNG, `use_videos=False`), which gives real image
  inputs **without** the video-codec stack.

> Part B ships a **synthetic camera** (a deterministic render of the joint state) so
> the whole pipeline runs end-to-end. It is a *scaffold*: for meaningful visuomotor
> results, replace it with **real recordings** via `lerobot-record` (a camera watching
> the arm). The code/commands are exactly what you'd use either way.

Docs: [installation](https://huggingface.co/docs/lerobot/en/installation) ·
[act](https://huggingface.co/docs/lerobot/en/act) ·
[il_robots](https://huggingface.co/docs/lerobot/en/il_robots) ·
[peft_training](https://huggingface.co/docs/lerobot/en/peft_training)

**Runtime:** Colab **GPU**.

## 0 · Setup — clone repo + install LeRobot (official)

In [ ]:
# RUN FIRST.  Colab GPU runtime.
import os, sys, subprocess, pathlib
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout.strip()
      or "No GPU - set Runtime > Change runtime type > GPU")

REPO_URL = "https://github.com/trishan9/D.R.O.N.A.git"   # <-- EDIT if you forked
def _is_repo(p): return pathlib.Path(p, "drona", "__init__.py").is_file()
repo = next((p for p in (".", "/content/D.R.O.N.A") if _is_repo(p)), None)
if repo is None:
    subprocess.run(["git","clone","--depth","1",REPO_URL,"/content/D.R.O.N.A"], check=True)
    repo = "/content/D.R.O.N.A"
os.chdir(repo); print("repo:", os.getcwd())

# Official install (docs/installation): base lerobot + `training` extra. ACT is
# in the base package. SmolVLA + PEFT extras are installed later in Part B.
subprocess.run([sys.executable,"-m","pip","install","-q","lerobot[training]"], check=True)

try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"] = tok; print("HF token loaded from Colab Secrets")
except Exception: pass

import lerobot; print("lerobot:", getattr(lerobot,"__version__","?")); print("setup complete")

## 1 · Get the gesture demonstrations

Upload **`drona_private_data.zip`** (contains `data/demonstrations/demonstrations.jsonl`
- 5,000 frames / 150 episodes / 6 gestures). The picker blocks until you choose it.

In [ ]:
import glob, pathlib, subprocess
DEMOS = pathlib.Path("data/demonstrations/demonstrations.jsonl")
if not DEMOS.exists():
    found = (glob.glob("drona_private_data.zip") + glob.glob("/content/drona_private_data.zip")
             + glob.glob("/kaggle/input/**/drona_private_data.zip", recursive=True))
    if not found:
        try:
            from google.colab import files
            print(">>> Upload drona_private_data.zip <<<"); up = files.upload()
            found = [n for n in up if n.endswith(".zip")]
        except ImportError: pass
    if found: subprocess.run(["unzip","-oq",found[0],"-d","."], check=True)
assert DEMOS.exists(), "upload drona_private_data.zip and re-run"

# shared loader: flat per-frame jsonl -> {episode_index: [frames...]}
import json, collections
ROWS = [json.loads(l) for l in open(DEMOS, encoding="utf-8")]
EPS = collections.defaultdict(list)
for r in ROWS: EPS[r["episode_index"]].append(r)
for ei in EPS: EPS[ei].sort(key=lambda r: r["frame_index"])
DOF, FPS = len(ROWS[0]["observation_state"]), 20
print(f"{len(ROWS)} frames / {len(EPS)} episodes / DOF={DOF}")

---
# Part A · State-only policies (default, correct for gestures)

### A.1 · Build a state-only LeRobotDataset (no camera, no video)

In [ ]:
import shutil, pathlib, numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ROOT_A = pathlib.Path("data/lerobot/drona_gestures")
if ROOT_A.exists(): shutil.rmtree(ROOT_A)
# ACT's validate_features() requires at least one IMAGE or an ENVIRONMENT_STATE
# feature - plain "observation.state" (robot proprioception) is not enough:
#   if not self.image_features and not self.env_state_feature: raise ValueError(...)
# lerobot maps the key "observation.environment_state" -> FeatureType.ENV. For a
# gesture with no external objects the arm configuration IS the environment
# state, so we publish the same joint vector under both keys.
FEATURES_A = {
    "observation.state": {"dtype":"float32","shape":(DOF,),"names":[f"joint_{i}" for i in range(DOF)]},
    "observation.environment_state": {"dtype":"float32","shape":(DOF,),
                                      "names":[f"joint_{i}" for i in range(DOF)]},
    "action":            {"dtype":"float32","shape":(DOF,),"names":[f"joint_{i}" for i in range(DOF)]},
}
ds = LeRobotDataset.create(repo_id="drona/gestures", fps=FPS, features=FEATURES_A,
                           root=str(ROOT_A), use_videos=False)
for ei in sorted(EPS):
    for fr in EPS[ei]:
        st = np.asarray(fr["observation_state"], np.float32)
        ds.add_frame({"observation.state": st,
                      "observation.environment_state": st,
                      "action": np.asarray(fr["action"], np.float32),
                      "task": fr["gesture_label"]})
    ds.save_episode()
ds.finalize()   # REQUIRED: writes the parquet footers/metadata;
                 # without it the dataset is unreadable by lerobot-train
print(f"state-only dataset -> {ROOT_A} ({len(EPS)} episodes)")

# Verify NOW - an invalid dataset otherwise fails only after lerobot-train spins
# up, and the traceback points at parquet internals rather than the real cause.
import json as _json
import pyarrow.parquet as _pq
_info = _json.load(open(ROOT_A / "meta" / "info.json", encoding="utf-8"))
_feats = list(_info.get("features", {}).keys())
assert 'observation.environment_state' in _feats, (
    f"observation.environment_state missing -> ACT rejects this dataset "
    f"(validate_features needs an image or environment_state). features={_feats}")
for _f in ROOT_A.rglob("*.parquet"):
    _pq.read_schema(_f)      # raises if finalize() did not write the footer
print("verified:", 'observation.environment_state', "present; parquet footers valid")


### A.2 · Train ACT (state-only)
Official `lerobot-train` CLI. `steps=2000` = smoke run; raise to 20000+ for the full model.

In [ ]:
import subprocess
def train(cmd):
    print(">", " ".join(cmd)); r = subprocess.run(cmd, text=True, capture_output=True)
    print(r.stdout[-2500:])
    if r.returncode: print("--- STDERR ---\n"+(r.stderr or "")[-4000:]); raise SystemExit("training failed")

train(["lerobot-train","--dataset.repo_id=drona/gestures","--dataset.root=data/lerobot/drona_gestures",
       "--policy.type=act","--policy.device=cuda","--batch_size=8","--steps=2000",
       "--output_dir=outputs/train/act_state","--job_name=act_state","--wandb.enable=false",
       "--policy.push_to_hub=false"])
print("ACT (state) done -> outputs/train/act_state")

### A.3 · Train Diffusion (state-only)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","lerobot[diffusion]"], check=True)
train(["lerobot-train","--dataset.repo_id=drona/gestures","--dataset.root=data/lerobot/drona_gestures",
       "--policy.type=diffusion","--policy.device=cuda","--batch_size=8","--steps=3000",
       "--output_dir=outputs/train/diffusion_state","--job_name=diffusion_state","--wandb.enable=false",
       "--policy.push_to_hub=false"])
print("Diffusion (state) done -> outputs/train/diffusion_state")

### A.4 · Evaluate — open-loop reproduction (action MSE + jerk, the C3 metric)

In [ ]:
import glob, json, numpy as np, torch
from lerobot.policies.act.modeling_act import ACTPolicy

def latest(base):
    c = sorted(glob.glob(f"{base}/checkpoints/*/pretrained_model")); return c[-1] if c else None
def jerk(a):
    a = np.asarray(a, np.float32); return float(np.mean(np.abs(np.diff(a,3,0)))) if len(a)>3 else float("nan")

by_g = {}                                   # held-out = last episode of each gesture
for ei in EPS: by_g.setdefault(EPS[ei][0]["gesture_label"], []).append(ei)
heldout = [max(v) for v in by_g.values()]
ck = latest("outputs/train/act_state"); print("ACT checkpoint:", ck)
try:
    p = ACTPolicy.from_pretrained(ck); p.eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; p.to(dev)
    mses, jp, jg = [], [], []
    for ei in heldout:
        gt = np.asarray([f["action"] for f in EPS[ei]], np.float32); pr=[]
        p.reset()
        for f in EPS[ei]:
            obs={"observation.state": torch.tensor(f["observation_state"],dtype=torch.float32,device=dev).unsqueeze(0)}
            with torch.no_grad(): pr.append(p.select_action(obs).squeeze(0).cpu().numpy())
        pr=np.asarray(pr,np.float32); mses.append(float(np.mean((pr-gt)**2))); jp.append(jerk(pr)); jg.append(jerk(gt))
    print(f"ACT open-loop  MSE={np.mean(mses):.4f}  jerk(pred)={np.nanmean(jp):.1f}  jerk(demo)={np.nanmean(jg):.1f}")
    json.dump({"action_mse":float(np.mean(mses)),"jerk_pred":float(np.nanmean(jp)),"jerk_gt":float(np.nanmean(jg))},
              open("outputs/act_state_eval.json","w"), indent=2)
except Exception as e:
    print(f"eval best-effort skipped: {type(e).__name__}: {str(e)[:150]}")

---
# Part B · Visuomotor (camera) + SmolVLA-LoRA

Adds a **camera observation** so we can train vision-conditioned ACT and
LoRA-fine-tune **SmolVLA** (a vision-language-action model). Frames are stored as
`dtype="image"` (PNG, `use_videos=False`) -> real images, **no video codec**.

> **Scaffold:** the camera here is a *synthetic render of the joint state* (6 bars),
> so the pipeline runs end-to-end. For real visuomotor results, record demos with a
> real camera via `lerobot-record` and point the dataset at them - the commands below
> are unchanged.

### B.1 · Build a camera dataset (synthetic render, `dtype=image`)

In [ ]:
import shutil, pathlib, numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

H = W = 96
def render(state):
    """Deterministic RGB render of the 6 joint angles as vertical bars (a stand-in
    for a real camera; correlates with state so a visuomotor policy can learn)."""
    img = np.zeros((H, W, 3), np.uint8); n = len(state); bw = W // n
    for i, v in enumerate(state):
        h = int((np.tanh(v) * 0.5 + 0.5) * (H - 1))
        img[H-1-h:, i*bw:(i+1)*bw, i % 3] = 220
    return img

ROOT_B = pathlib.Path("data/lerobot/drona_gestures_cam")
if ROOT_B.exists(): shutil.rmtree(ROOT_B)
FEATURES_B = {
    "observation.state": {"dtype":"float32","shape":(DOF,),"names":[f"joint_{i}" for i in range(DOF)]},
    "observation.images.front": {"dtype":"image","shape":(H,W,3),"names":["height","width","channel"]},
    "action": {"dtype":"float32","shape":(DOF,),"names":[f"joint_{i}" for i in range(DOF)]},
}
dsb = LeRobotDataset.create(repo_id="drona/gestures_cam", fps=FPS, features=FEATURES_B,
                            root=str(ROOT_B), use_videos=False)   # image dtype => PNG, no codec
for ei in sorted(EPS):
    for fr in EPS[ei]:
        st = np.asarray(fr["observation_state"], np.float32)
        dsb.add_frame({"observation.state": st,
                       "observation.images.front": render(st),
                       "action": np.asarray(fr["action"], np.float32),
                       "task": fr["gesture_label"]})
    dsb.save_episode()
dsb.finalize()   # REQUIRED: writes the parquet footers/metadata;
                 # without it the dataset is unreadable by lerobot-train
print(f"camera dataset -> {ROOT_B} ({len(EPS)} episodes, {H}x{W} PNG frames)")

# Verify NOW - an invalid dataset otherwise fails only after lerobot-train spins
# up, and the traceback points at parquet internals rather than the real cause.
import json as _json
import pyarrow.parquet as _pq
_info = _json.load(open(ROOT_B / "meta" / "info.json", encoding="utf-8"))
_feats = list(_info.get("features", {}).keys())
assert 'observation.images.front' in _feats, (
    f"observation.images.front missing -> ACT rejects this dataset "
    f"(validate_features needs an image or environment_state). features={_feats}")
for _f in ROOT_B.rglob("*.parquet"):
    _pq.read_schema(_f)      # raises if finalize() did not write the footer
print("verified:", 'observation.images.front', "present; parquet footers valid")


### B.2 · Train visuomotor ACT (with camera)

In [ ]:
train(["lerobot-train","--dataset.repo_id=drona/gestures_cam","--dataset.root=data/lerobot/drona_gestures_cam",
       "--policy.type=act","--policy.device=cuda","--batch_size=8","--steps=2000",
       "--output_dir=outputs/train/act_cam","--job_name=act_cam","--wandb.enable=false",
       "--policy.push_to_hub=false"])
print("visuomotor ACT done -> outputs/train/act_cam")

### B.3 · Fine-tune SmolVLA with LoRA / PEFT

Per [peft_training](https://huggingface.co/docs/lerobot/en/peft_training): install the
`smolvla` + `peft` extras and fine-tune `lerobot/smolvla_base` with
`--peft.method_type=LORA`. SmolVLA is a vision-language-action model, so it uses the
camera + the per-frame `task` (gesture label) as its language instruction + the state.
LoRA LR is ~10x the full-FT LR (docs).

In [ ]:
# Follows the official SmolVLA guide (docs/smolvla) for setup, plus the
# peft_training guide's LoRA flags. Sequence matters:
import subprocess, sys, pathlib, re, importlib.util
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "lerobot[smolvla]", "lerobot[peft]"], check=True)
# PEFT's LoRA dispatch imports torchao and requires > 0.16 (Colab ships 0.10).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "torchao>=0.16", "peft>=0.18,<1.0"], check=True)

# The torchao upgrade drags in a diffusers build with a real bug: its
# quantizers/torchao/torchao_quantizer.py calls logger.warning() at module load
# (line ~107) but only DEFINES `logger` further down the file - so importing
# lerobot (factory -> fastwam -> diffusers) raises NameError before training.
# Fix: inject a module-level logger BEFORE the function that uses it. Marker makes
# it idempotent. (Version pinning can't dodge it - the broken build is in range.)
_spec = importlib.util.find_spec("diffusers")
if _spec:
    _f = pathlib.Path(_spec.origin).parent / "quantizers" / "torchao" / "torchao_quantizer.py"
    if _f.exists():
        _src = _f.read_text()
        if "_DRONA_LOGGER_FIX" not in _src:
            _patched = re.sub(
                r"(?m)^def _update_torch_safe_globals",
                "import logging as _drona_log  # _DRONA_LOGGER_FIX\n"
                "logger = _drona_log.getLogger('diffusers.torchao')\n\n\n"
                "def _update_torch_safe_globals",
                _src, count=1)
            if _patched != _src:
                _f.write_text(_patched)
                print("patched diffusers torchao_quantizer logger bug ->", _f)
            else:
                print("WARNING: could not locate _update_torch_safe_globals to patch")
        else:
            print("diffusers logger fix already applied")

train(["lerobot-train",
       "--policy.path=lerobot/smolvla_base",
       "--dataset.repo_id=drona/gestures_cam",
       "--dataset.root=data/lerobot/drona_gestures_cam",
       "--policy.device=cuda",
       "--batch_size=4", "--steps=2000",
       "--policy.optimizer_lr=1e-3", "--policy.scheduler_decay_lr=1e-4",
       # smolvla_base expects cameras camera1/2/3; the validator passes when the
       # dataset's cameras are a subset, so map our single camera to camera1.
       '--rename_map={"observation.images.front": "observation.images.camera1"}',
       "--peft.method_type=LORA", "--peft.r=64", "--peft.lora_alpha=64",
       "--output_dir=outputs/train/smolvla_lora", "--job_name=smolvla_lora",
       "--wandb.enable=false", "--policy.push_to_hub=false"])
print("SmolVLA-LoRA done -> outputs/train/smolvla_lora")


## 2 · Save all checkpoints to Google Drive

In [ ]:
import shutil, pathlib
try:
    from google.colab import drive; drive.mount("/content/drive")
    dst = pathlib.Path("/content/drive/MyDrive/drona_training/lerobot"); dst.mkdir(parents=True, exist_ok=True)
    for src in ("outputs/train/act_state","outputs/train/diffusion_state","outputs/train/act_cam",
                "outputs/train/smolvla_lora","outputs/act_state_eval.json"):
        s = pathlib.Path(src)
        if s.exists():
            d = dst / s.name
            shutil.copytree(s, d, dirs_exist_ok=True) if s.is_dir() else shutil.copy2(s, d)
            print("saved ->", d)
    print("\nLeRobot policies persisted to Drive/drona_training/lerobot")
except Exception as e:
    print("Drive unavailable:", e, "- download outputs/ before the session ends")